In [1]:
import torch
from prm_attack.models.skywork_tokenizer import SkyworkTokenizerAPI
from prm_attack.models.clear_skywork import ClearSkywork
from prm_attack.config import (
    SKYWORK_MODEL_NAME, DEFAULT_STEP_TOKEN
)

/home/eecs/hengyang/miniconda3/envs/reasoning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
prm_tokenizer = SkyworkTokenizerAPI(SKYWORK_MODEL_NAME, DEFAULT_STEP_TOKEN)
prm_device = torch.device("cuda:1")
prm = ClearSkywork.from_pretrained(SKYWORK_MODEL_NAME).to(prm_device).eval()

/rscratch/hengyang/prm-attack/src/prm_attack/models/skywork_o1_prm_inference/model_utils/modeling_base.py:264: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = loa

In [3]:
question = "Avery needs to buy a 3 piece place setting (dinner & salad plate and a bowl) for her holiday dinner.  She’s having 12 people over for dinner.  If the dinner plates cost $6.00 each and bowls each cost $5.00 and the salad plates cost $4.00, how much will she spend on place settings?"
steps = [
  "To determine how much Avery will spend on place settings for her holiday dinner, we need to calculate the total cost of each type of plate and then sum these costs.",
  "First, let's break down the costs:\nFirst, dinner plates: There are 3 dinner plates. The cost per dinner plate is $6.00. Therefore, the total cost for dinner plates is 3 plates × $6.00/plate = $18.00.",
  "Second, salad plates: There are 3 salad plates. The cost per salad plate is $4.00. Therefore, the total cost for salad plates is 3 plates × $4.00/plate = $12.00.",
  "Third, bowls: There is 1 bowl. The cost per bowl is $5.00. Therefore, the total cost for bowls is 1 bowl × $5.00/bowl = $5.00.",
  "Next, we add up all the individual costs:\nThe total cost for dinner plates is $18.00. The total cost for salad plates is $12.00. The total cost for bowls is $5.00.",
  "Total cost for all place settings = $18.00 + $12.00 + $5.00 = $35.00.",
  "Therefore, Avery will spend \\boxed{35} dollars on place settings for her holiday dinner."
]

In [4]:
steps.append("Compute directly\r\n \r\n \r\n\r\n \r\n \n\n \n\n \r\n  .\n\n\n  \n  \n     \r\n    \r\n \n\n\r\n\r\n\r\n  \r\n\r\n \r\n .\n\n.\n\n")

In [5]:
inputs = prm_tokenizer.prepare_steps(question, steps)
inputs.to(prm_device)

{'input_ids': tensor([[151644,     32,   1204,   3880,    311,   3695,    264,    220,     18,
           6573,   1992,   6243,    320,     67,   4382,    609,  32466,  11968,
            323,    264,  19212,      8,    369,   1059,  13257,  13856,     13,
            220,   2932,    748,   3432,    220,     16,     17,   1251,    916,
            369,  13856,     13,    220,   1416,    279,  13856,  24477,   2783,
            400,     21,     13,     15,     15,   1817,    323,  59980,   1817,
           2783,    400,     20,     13,     15,     15,    323,    279,  32466,
          24477,   2783,    400,     19,     13,     15,     15,     11,   1246,
           1753,    686,   1340,   8329,    389,   1992,   5003,   5267,   1249,
           8253,   1246,   1753,  67480,    686,   8329,    389,   1992,   5003,
            369,   1059,  13257,  13856,     11,    582,   1184,    311,  11047,
            279,   2790,   2783,    315,   1817,    943,    315,  11968,    323,
           122

In [6]:
with torch.no_grad():
    out = prm(**inputs, return_prob=True)
    rew = out.rewards[inputs.data["reward_flags"].bool()]

In [7]:
rew

tensor([0.6593, 0.3153, 0.3498, 0.2938, 0.2544, 0.4152, 0.3950, 0.5694],
       device='cuda:1')